# Measurements preparations

## Preamble

In [101]:
import os
from glob import glob
from datetime import datetime
import csv
import xml.etree.ElementTree as ET
import math

import pandas as pd
from libnmap.parser import NmapParser
import ipaddress

## IPv4

### NMap allowlist

In [59]:
ts = datetime(2025, 8, 27)  # date when ISI hitlist was made

In [60]:
_anycast_ipv4_pdf = pd.read_csv(f"anycast_census_{ts.year}_{ts.month:02d}_{ts.day:02d}_v4.csv")
anycast_ipv4_pdf = _anycast_ipv4_pdf[_anycast_ipv4_pdf["GCD_ICMPv4"] > 1].copy()
display(anycast_ipv4_pdf.head(5))
print(len(anycast_ipv4_pdf))

,prefix,AB_ICMPv4,AB_TCPv4,AB_DNSv4,GCD_ICMPv4,GCD_TCPv4,partial,backing_prefix,ASN,locations
0,1.0.0.0/24,29,29,29,75,30,False,1.0.0.0/24,13335,"[{'city': 'Honolulu', 'code_country': 'US', 'i..."
1,1.1.1.0/24,29,29,0,74,31,False,1.1.1.0/24,13335,"[{'city': 'Bangalore', 'code_country': 'IN', '..."
3,1.10.10.0/24,2,0,2,2,0,False,1.10.10.0/24,148000,"[{'city': 'Mumbai', 'code_country': 'IN', 'id'..."
4,1.12.0.0/24,5,0,5,5,0,False,1.12.0.0/20,132203_45090,"[{'city': 'Cologne', 'code_country': 'DE', 'id..."
5,1.12.12.0/24,3,1,0,3,1,False,1.12.0.0/20,132203_45090,"[{'city': 'Hong Kong', 'code_country': 'HK', '..."


13361


In [61]:
ipv4_pdf = pd.read_csv(f"internet_address_verfploeter_hitlist_it113w-{ts.year}{ts.month:02d}{ts.day:02d}.csv", header=None)
ipv4_pdf.columns = ["ipv4"]

display(ipv4_pdf.head(5))
print(len(ipv4_pdf))

,ipv4
0,1.0.0.0
1,1.0.4.1
2,1.0.5.1
3,1.0.6.1
4,1.0.7.1


5788894


In [62]:
ipv4_pdf["prefix"] = ipv4_pdf["ipv4"].apply(lambda x: ".".join(x.split(".")[:3]) + ".0/24")

display(ipv4_pdf.head(5))

,ipv4,prefix
0,1.0.0.0,1.0.0.0/24
1,1.0.4.1,1.0.4.0/24
2,1.0.5.1,1.0.5.0/24
3,1.0.6.1,1.0.6.0/24
4,1.0.7.1,1.0.7.0/24


In [63]:
responsive_ipv4_pdf = anycast_ipv4_pdf.merge(ipv4_pdf, on="prefix", how="inner")

display(responsive_ipv4_pdf.head(5))
print(len(responsive_ipv4_pdf))

,prefix,AB_ICMPv4,AB_TCPv4,AB_DNSv4,GCD_ICMPv4,GCD_TCPv4,partial,backing_prefix,ASN,locations,ipv4
0,1.0.0.0/24,29,29,29,75,30,False,1.0.0.0/24,13335,"[{'city': 'Honolulu', 'code_country': 'US', 'i...",1.0.0.0
1,1.1.1.0/24,29,29,0,74,31,False,1.1.1.0/24,13335,"[{'city': 'Bangalore', 'code_country': 'IN', '...",1.1.1.0
2,1.10.10.0/24,2,0,2,2,0,False,1.10.10.0/24,148000,"[{'city': 'Mumbai', 'code_country': 'IN', 'id'...",1.10.10.10
3,1.12.0.0/24,5,0,5,5,0,False,1.12.0.0/20,132203_45090,"[{'city': 'Cologne', 'code_country': 'DE', 'id...",1.12.0.0
4,1.12.12.0/24,3,1,0,3,1,False,1.12.0.0/20,132203_45090,"[{'city': 'Hong Kong', 'code_country': 'HK', '...",1.12.12.12


13355


In [ ]:
responsive_ipv4 = responsive_ipv4_pdf.drop_duplicates(subset=["ipv4"])["ipv4"].copy()
responsive_ipv4.to_csv(f"../input/nmap/responsive_anycast_ipv4_addresses_{ts.year}{ts.month:02d}{ts.day:02d}.csv", index=False, header=False)

In [ ]:
# dividing into shards to run nmap in parallel if necessary
nshards = 30

for shard in range(nshards):
    shard_pdf = responsive_ipv4.iloc[shard::nshards]
    shard_pdf.to_csv(f"../input/nmap/sharding/responsive_anycast_ipv4_addresses_shard_{shard+1}_of_{nshards}_{ts.year}{ts.month:02d}{ts.day:02d}.csv", index=False, header=False)

In [ ]:
# confirming that shards don't overlap

all_ips = set()
for shard in range(nshards):
    shard_pdf = pd.read_csv(f"input/nmap/sharding/responsive_anycast_ipv4_addresses_shard_{shard+1}_of_{nshards}_{ts.year}{ts.month:02d}{ts.day:02d}.csv", header=None)
    shard_pdf.columns = ["ipv4"]
    shard_ips = set(shard_pdf["ipv4"].tolist())
    intersection = all_ips.intersection(shard_ips)
    assert len(intersection) == 0, f"Overlap found in shard {shard+1}, {intersection}"
    all_ips.update(shard_ips)

### ZMap allowlist

In [47]:
ts = datetime(2025, 12, 2)

In [ ]:
_anycast_ipv4_pdf = pd.read_csv(f"../anycast_census_{ts.year}_{ts.month:02d}_{ts.day:02d}_v4.csv")
anycast_ipv4_pdf = _anycast_ipv4_pdf[_anycast_ipv4_pdf["GCD_ICMPv4"] > 1].copy()
display(anycast_ipv4_pdf.head(5))
print(len(anycast_ipv4_pdf))
anycast_ipv4_pdf["prefix"].to_csv(f"../input/zmap/anycast_prefixes_{ts.year}_{ts.month:02d}_{ts.day:02d}_v4.csv", index=False, header=False)

,prefix,AB_ICMPv4,AB_TCPv4,AB_DNSv4,GCD_ICMPv4,GCD_TCPv4,partial,backing_prefix,ASN,locations
0,1.0.0.0/24,27,29,28,67,30,False,1.0.0.0/24,13335,"[{'city': 'Bangalore', 'country_code': 'IN', '..."
2,1.1.1.0/24,27,29,28,65,31,False,1.1.1.0/24,13335,"[{'city': 'Bangalore', 'country_code': 'IN', '..."
5,1.10.10.0/24,3,0,3,2,0,False,1.10.10.0/24,148000,"[{'city': 'Mumbai', 'country_code': 'IN', 'id'..."
6,1.12.0.0/24,4,0,3,5,0,False,1.12.0.0/20,132203,"[{'city': 'Singapore', 'country_code': 'SG', '..."
7,1.12.12.0/24,3,1,0,3,1,False,1.12.0.0/20,132203,"[{'city': 'Hong Kong', 'country_code': 'HK', '..."


13994


#### ZMap ports

In [ ]:
xml_files = glob("../results/nmap/*.xml")

ports = set()
for xml_file in xml_files:
    print(f"Processing {xml_file}...")
    report = NmapParser.parse_fromfile(xml_file)
    for host in report.hosts:
        for svc in host.services:
            # tcpwrapped are hosts that deliberately closed the TCP connection
            # https://secwiki.org/w/FAQ_tcpwrapped
            if svc.service in ["tcpwrapped"]:
                continue
            if svc.state == "open":
                ports.add(svc.port)


print(f"Discovered {len(ports)} open ports")

Processing ../results/nmap/nmap_20251202093456_.xml...
Discovered 1997 open ports


In [42]:
# tab delimited file
nmap_services_pdf = pd.read_csv("../nmap-services", delimiter="\t")
nmap_services_pdf.columns = ["service_name", "portnum_protocol", "open_frequency", "optional_comments"]
nmap_services_pdf.sort_values(by=["open_frequency"], ascending=False, inplace=True)
nmap_services_pdf["portnum"] = nmap_services_pdf["portnum_protocol"].apply(lambda x: int(x.split("/")[0]))
nmap_services_pdf["protocol"] = nmap_services_pdf["portnum_protocol"].apply(lambda x: x.split("/")[1])

default_nmap_ports_pdf = nmap_services_pdf[nmap_services_pdf["protocol"] == "tcp"].head(2000).copy()

default_nmap_ports = set(default_nmap_ports_pdf["portnum"].tolist())

scanned_but_missing_default = ports.difference(default_nmap_ports)
print(len(scanned_but_missing_default), "ports from measurements are not in default nmap top 2000 ports:", scanned_but_missing_default)

default_but_missing_ports = default_nmap_ports.difference(ports)
print(len(default_but_missing_ports), "default nmap ports are missing in our scan results:", default_but_missing_ports)

316 ports from measurements are not in default nmap top 2000 ports: {1, 2070, 4161, 2124, 2134, 2142, 4192, 2148, 2150, 2187, 4243, 2197, 2203, 4262, 2224, 2250, 2253, 2261, 2262, 2265, 2269, 2270, 2271, 2280, 2291, 2292, 4342, 2300, 2302, 2304, 4355, 4356, 4357, 4358, 2312, 2313, 2325, 2326, 2330, 2335, 2340, 2371, 2372, 2375, 2376, 2391, 2418, 2425, 2435, 2436, 2438, 2439, 2449, 2456, 2463, 2472, 2550, 2551, 2580, 2583, 2691, 4767, 4770, 2723, 4771, 4778, 4793, 4800, 4819, 2804, 2806, 4859, 2812, 4860, 4876, 4877, 4881, 2847, 2850, 4903, 4912, 2882, 4931, 2888, 2889, 2898, 2901, 2902, 2908, 2930, 2957, 2958, 2973, 2984, 2987, 2988, 2991, 2997, 3002, 3014, 3023, 3057, 3062, 3063, 3080, 3089, 3118, 1101, 1116, 1118, 1125, 1128, 1134, 1135, 1136, 5233, 5234, 5235, 3190, 1143, 1144, 1150, 1153, 5250, 1156, 1157, 5252, 1159, 1162, 5259, 5261, 1167, 1168, 1173, 1176, 1179, 1180, 1182, 1184, 1188, 1190, 1191, 1194, 1195, 1196, 5291, 1200, 1204, 1207, 1208, 1209, 1210, 1211, 1215, 1221, 1223

nmap scans top 1000 ports per protocol: https://nmap.org/book/man-port-specification.html

In [52]:
def parse_nmap_ports(port_string):
    """
    CHATGPT GENERATED FUNCTION
    Parse an Nmap port range string like:
    '1,3-4,6-7,9,13,17-20'
    and return a sorted list of all port numbers.
    """
    ports = set()

    for part in port_string.split(","):
        part = part.strip()
        
        # Range: e.g., "300-305"
        if "-" in part:
            start, end = part.split("-")
            start = int(start)
            end = int(end)
            ports.update(range(start, end + 1))
        
        # Single port
        else:
            ports.add(int(part))

    return set(sorted(ports))


# nmap --top-ports 2000 localhost -v -oG -
with open("nmap_localhost_2k.txt", "rt", encoding="utf-8") as f:
    result2k = f.read()

# comparing with real scans of 9 hosts...
localhost_ports = parse_nmap_ports(result2k)

print(f"Nmap default top ports count: {len(localhost_ports)}")

print(len(localhost_ports.difference(ports)))

Nmap default top ports count: 2000
3


In [ ]:
# nmap localhost -v -oG -
with open("nmap_localhost_1k.txt", "rt", encoding="utf-8") as f:
    result1k = f.read()

localhost_ports = parse_nmap_ports(result1k)

print(f"Nmap default top ports count: {len(localhost_ports)}")

with open("../input/zmap/nmap_ports", "w") as f:
    for item in localhost_ports:
        f.write(f"{item}\n")

Nmap default top ports count: 1000


### ZGrab2 allowlist

In [138]:
# https://github.com/zmap/zgrab2/
service_names = [s.lower() for s in ["AMQP", "BACnet", "Banner", "DNP", 
             "Fox", "FTP", "HTTP", "IMAP", "IPP",
             "JARM", "Sieve", "Memcached",
             "Modbus", "MongoDB", "MQTT", "MSSQL",
             "MySQL", "NTP", "Oracle", "POP3",
             "PostgreSQL", "PPTP", "Redis", "Siemens",
             "SMB", "SMTP", "SOCKS5", "SSH", "Telnet"]]

Finding out which protocols we scanned that matches ZGrab2

In [139]:
# nmap-services comes from /opt/homebew/share/nmap/nmap-services file (default installation of nmap on macOS)
nmap_services_pdf = pd.read_csv("../nmap-services", delimiter="\t")
nmap_services_pdf.columns = ["service_name", "portnum_protocol", "open_frequency", "optional_comments"]
nmap_services_pdf.sort_values(by=["open_frequency"], ascending=False, inplace=True)
nmap_services_pdf["portnum"] = nmap_services_pdf["portnum_protocol"].apply(lambda x: int(x.split("/")[0]))
nmap_services_pdf["protocol"] = nmap_services_pdf["portnum_protocol"].apply(lambda x: x.split("/")[1])

def in_zgrab2(x):
    for service_name in service_names:
        if service_name in x.lower().strip():
            return True
    return False

nmap_services_pdf["in_zgrab2"] = nmap_services_pdf["service_name"].apply(lambda x: in_zgrab2(x))

In [140]:
# nmap ports comes from a dry-run of nmap on localhost
with open("../input/zmap/nmap_ports", "rt", encoding="utf-8") as f:
    result = f.read().splitlines()

nmap_ports_pdf = pd.DataFrame({"portnum": [int(line.strip()) for line in result]})
print(len(nmap_ports_pdf))
# join to know if ports are in zgrab2
joined_pdf = nmap_ports_pdf.merge(nmap_services_pdf, on="portnum", how="inner")

joined_pdf[joined_pdf["in_zgrab2"] == True]

1000


,portnum,service_name,portnum_protocol,open_frequency,optional_comments,protocol,in_zgrab2
39,20,ftp-data,20/udp,0.001878,# File Transfer [Default Data],udp,True
40,20,ftp-data,20/tcp,0.001079,# File Transfer [Default Data],tcp,True
41,20,ftp-data,20/sctp,0.000000,# File Transfer [Default Data],sctp,True
42,21,ftp,21/tcp,0.197667,# File Transfer [Control],tcp,True
43,21,ftp,21/udp,0.004844,# File Transfer [Control],udp,True
...,...,...,...,...,...,...,...
1673,8080,http-proxy,8080/tcp,0.042052,# http-alt | Common HTTP proxy/second web serv...,tcp,True
1674,8080,http-alt,8080/udp,0.000000,# HTTP Alternate (see port 80),udp,True
1689,8088,radan-http,8088/tcp,0.000608,# Radan HTTP,tcp,True
1690,8088,radan-http,8088/udp,0.000000,# Radan HTTP,udp,True


In [141]:
udp_ports = joined_pdf[joined_pdf["protocol"] != "tcp"]["portnum"].tolist()
tcp_ports = nmap_ports_pdf["portnum"].tolist()

for u in udp_ports:
    if u not in tcp_ports:
        print(u)

# print nothing means we don't overlook any port. We scan all tcp ports as should.

In [142]:
def needs_tls(s, c):
    if "tls" in s.lower() or "ssl" in s.lower():
        return True
    if c is None or pd.isna(c):
        return False
    if "ssl" in c.lower() or "tls" in c.lower():
        return True
    return False

joined_pdf["needs_tls"] = joined_pdf[["service_name", "optional_comments"]].apply(lambda x: needs_tls(x["service_name"], x["optional_comments"]), axis=1)

joined_pdf[(joined_pdf["needs_tls"] == True) & (joined_pdf["in_zgrab2"] == True)].drop_duplicates(subset=["portnum"])[["portnum", "service_name", "optional_comments"]]

,portnum,service_name,optional_comments
337,443,https,# secure http (SSL)
361,465,smtps,# submissions | igmpv3lite | urd | smtp protoc...
734,990,ftps,"# ftp protocol, control, over TLS/SSL"
737,992,telnets,# telnet protocol over TLS/SSL
739,993,imaps,# imap4 protocol over TLS/SSL | IMAP over TLS ...
741,995,pop3s,# POP3 protocol over TLS/SSL | pop3 protocol o...


In [143]:
joined_pdf[(joined_pdf["needs_tls"] == True) & (joined_pdf["in_zgrab2"] == False)].drop_duplicates(subset=["portnum"])[["portnum", "service_name", "optional_comments"]]

,portnum,service_name,optional_comments
432,6699,napster,# babel-dtls | Napster File (MP3) sharing sof...
434,563,snews,# nntps | nntp protocol over TLS/SSL (was snntp)
464,636,ldapssl,# ldaps | LDAP over SSL | ldap protocol over T...
494,15002,onep-tls,# Open Network Environment TLS
573,9001,etlservicemgr,# ETL Service Manager
715,5061,sip-tls,NaN
988,1131,caspssl,# CAC App Service Protocol Encripted
1006,3077,orbix-loc-ssl,# Orbix 2000 Locator SSL
1093,3269,globalcatLDAPssl,# msft-gc-ssl | Global Catalog LDAP over ssl |...
1459,3766,sitewatch-s,# SSL e-watch sitewatch server


In [144]:
joined_pdf[(joined_pdf["needs_tls"] == False) & (joined_pdf["in_zgrab2"] == True)].drop_duplicates(subset=["portnum"])[["portnum", "service_name", "optional_comments"]]

,portnum,service_name,optional_comments
39,20,ftp-data,# File Transfer [Default Data]
42,21,ftp,# File Transfer [Control]
45,22,ssh,# Secure Shell Login
48,23,telnet,NaN
52,25,smtp,# Simple Mail Transfer
54,26,rsftp,NaN
102,2121,ccproxy-ftp,# scientia-ssdb | CCProxy FTP Proxy | SCIENTIA...
108,80,http,# World Wide Web HTTP
141,106,pop3pw,# 3com-tsmux | Eudora compatible PW changer | ...
148,110,pop3,# PostOffice V.3 | Post Office Protocol - Vers...


In [ ]:
# ports that zgrab knows but we did not scan
joined_pdf[joined_pdf["portnum"].isin([5445, 27017, 27018, 27019, 1883, 6379, 28240, 4190, 11211, 4319, 19999, 5671, 5672, 47808])]

,portnum,service_name,portnum_protocol,open_frequency,optional_comments,protocol,in_zgrab2,needs_tls


In [ ]:
def store_zgrab_input(dataset):
    # zmap v4 files are like: zmap_465_20251127123220.csv
    zmap_files = glob(f"../results/zmap/dataset={dataset}/**/zmap_*.csv", recursive=True)

    zmap_pdf = pd.DataFrame(columns=["saddr", "port"])
    for zmap_file in zmap_files:
        # extracting the port from the file name
        port = int(zmap_file.split("_")[-2])
        if port == 6699:
            # this is an UDP port
            continue
        zmap_port_pdf = pd.read_csv(zmap_file)
        zmap_port_pdf["port"] = port
        zmap_pdf = pd.concat([zmap_pdf, zmap_port_pdf[["saddr", "port"]]], ignore_index=True)


    zmap_pdf["domain"] = ""
    def _set_tag(port):
        # set the trigger of services zgrab knows and uses TLS
        if port in [443, 2381, 16993, 6789, 7443, 5989, 8443, 2381, 16993, 6789, 7443, 5989]:
            return "https"
        if port  == 465:
            return "smtps"
        if port == 990:
            return "ftps"
        if port == 992:
            return "telnettls"
        if port == 993:
            return "imaps"
        if port == 995:
            return "pop3s"

        # set the trigger of services zgrab doesn't know and uses TLS
        # 3995: https://learn.microsoft.com/en-us/iis/manage/remote-administration/remote-administration-for-iis-manager
        if port in [563, 636, 15002, 9001, 5061, 1131, 3077, 3269, 3766, 5986, 3995]:
            return "bannertls"

        # set the trigger of services zgrab knows and doesn't use TLS
        if port in [20, 21, 26, 2121, 2811, 8021]:
            return "ftp"
        if port == 22:
            return "ssh"
        if port == 23:
            return "telnet"
        if port == 25:
            return "smtp"
        if port in [80, 280, 593, 16992, 6788, 4848, 777, 808, 1183, 3128, 7627, 5800, 5801, 5802, 8000, 8008, 5988, 8080, 8088]:
            return "http"
        if port == 110:
            return "pop3"
        if port == 143:
            return "imap"
        if port == 631:
            return "ipp"
        if port in [1186, 3306, 1862]:
            return "mysql"
        if port == 5432:
            return "postgresql"
        if port in [1521, 2005]:
            return "oracle"
        if port == 20000:
            return "dnp3"
        if port == 1723:
            return "pptp"

        # ports 106, 119, 3920, and others go here...
        # default to banner
        return "banner"


    zmap_pdf["tag"] = zmap_pdf["port"].apply(lambda x: _set_tag(x))
    zmap_pdf[["saddr", "domain", "tag", "port"]].to_csv(f"../input/zgrab/zgrab_input_{dataset}.csv", index=False, header=False)

store_zgrab_input("tcp-anycast")

27


### NMap allowlist -- old pipeline

In [44]:
def create_allowlist_from_zmap(dataset, ts, output_filename):
    # zmap v4 files are like: zmap_465_20251127123220_v4.csv
    zmap_v4_files = glob(f"results/zmap/dataset={dataset}/**/zmap_*.csv", recursive=True)

    zmap_v4_pdf = pd.DataFrame(columns=["saddr", "port"])
    for zmap_v4_file in zmap_v4_files:
        # extracting the port from the file name
        port = int(zmap_v4_file.split("_")[-2])
        zmap_port_v4_pdf = pd.read_csv(zmap_v4_file)
        zmap_port_v4_pdf["port"] = port
        zmap_v4_pdf = pd.concat([zmap_v4_pdf, zmap_port_v4_pdf[["saddr", "port"]]], ignore_index=True)

    zmap_v4_pdf.drop_duplicates(["saddr"])["saddr"].to_csv(f"input/nmap/{output_filename}_{ts.year}{ts.month:02d}{ts.day:02d}.csv", index=False, header=False)

In [ ]:
ts = datetime(2025, 11, 28)  # date when ZMap scan was done
create_allowlist_from_zmap("tcp-anycast", ts, "zmap_responsive_tcp_ipv4_addresses")

### QUIC allowlist

In [45]:
ts = datetime(2025, 11, 28)  # date when ZMap UDP scan was done
create_allowlist_from_zmap("udp-anycast", ts, "zmap_responsive_udp_ipv4_addresses")

## IPv6

### ZMap allowlist -- unused

In [2]:
ts_v6 = datetime(2025, 11, 26)

In [ ]:
_anycast_ipv6_pdf = pd.read_csv(f"anycast_census_{ts_v6.year}_{ts_v6.month:02d}_{ts_v6.day:02d}_v6.csv")
anycast_ipv6_pdf = _anycast_ipv6_pdf[_anycast_ipv6_pdf["GCD_ICMPv6"] > 1].copy()
display(anycast_ipv6_pdf.head(5))
print(len(anycast_ipv6_pdf))
anycast_ipv6_pdf["prefix"].to_csv(f"input/anycast_prefixes_{ts_v6.year}_{ts_v6.month:02d}_{ts_v6.day:02d}_v6.csv", index=False, header=False)

,prefix,AB_ICMPv6,AB_TCPv6,AB_DNSv6,GCD_ICMPv6,GCD_TCPv6,backing_prefix,ASN,locations
0,2001:1201::/48,2,2,2,2,1,2001:1201::/48,28498,"[{'city': 'Querétaro', 'country_code': 'MX', '..."
11,2001:1258::/48,2,2,2,2,2,2001:1258::/32,28499,"[{'city': 'Querétaro', 'country_code': 'MX', '..."
13,2001:12f8:2::/48,3,0,3,3,0,2001:12f8:2::/48,12136,"[{'city': 'Singapore', 'country_code': 'SG', '..."
14,2001:12f8:4::/48,4,0,4,5,0,2001:12f8:4::/48,11644,"[{'city': None, 'country_code': None, 'id': 'N..."
15,2001:12f8:8::/48,3,0,3,3,0,2001:12f8:8::/48,10906,"[{'city': 'St Petersburg', 'country_code': 'RU..."


12821


### NMap allowlist

In [9]:
ts_v6 = datetime(2025, 8, 18)

In [10]:
_anycast_ipv6_pdf = pd.read_csv(f"anycast_census_{ts_v6.year}_{ts_v6.month:02d}_{ts_v6.day:02d}_v6.csv")
anycast_ipv6_pdf = _anycast_ipv6_pdf[_anycast_ipv6_pdf["GCD_ICMPv6"] > 1].copy()
display(anycast_ipv6_pdf.head(5))
print(len(anycast_ipv6_pdf))

,prefix,AB_ICMPv6,AB_TCPv6,AB_DNSv6,GCD_ICMPv6,GCD_TCPv6,backing_prefix,ASN,locations
0,2001:1201:10::/48,1,1,1,3,1,2001:1201:10::/48,27661,"[{'city': 'Vancouver', 'code_country': 'CA', '..."
1,2001:1201::/48,2,2,2,3,2,2001:1201::/48,28498,"[{'city': 'Monterrey', 'code_country': 'MX', '..."
6,2001:1250:a000::/48,3,2,3,2,1,2001:1250:a000::/44,28511_22894,"[{'city': 'Monterrey', 'code_country': 'MX', '..."
8,2001:1250:c000::/48,3,2,3,2,1,2001:1250:c000::/44,28511_22894,"[{'city': 'Monterrey', 'code_country': 'MX', '..."
10,2001:1258::/48,2,2,2,3,2,2001:1258::/32,28499,"[{'city': 'Querétaro', 'code_country': 'MX', '..."


6019


In [11]:
ipv6_pdf = pd.read_csv(f"v6_{ts_v6.year}{ts_v6.month:02d}{ts_v6.day:02d}.csv", header=None)

ipv6_pdf.columns = ["ipv6"]
display(ipv6_pdf.head(5))
print(len(ipv6_pdf))

,ipv6
0,2620:10a:80ac::45
1,2001:678:2c:0:194:0:28:7
2,2001:678:78:42:ad::53
3,2001:dce:2000:2::130
4,2001:dce:7000:2::130


6234585


In [12]:
def is_v6_in_prefix(ipv6, prefix):
    try:
        ipv6_addr = ipaddress.IPv6Address(ipv6)
        ipv6_net = ipaddress.IPv6Network(prefix)
        return ipv6_addr in ipv6_net
    except ValueError:
        return False


def ipv6_to_prefix48(ip):
    try:
        # Create a /48 network that contains this IP, non-strict so host bits allowed
        net = ipaddress.IPv6Network(f"{ip}/48", strict=False)
        # Return in same format as your prefix column, e.g. "2001:1201:10::/48"
        return net.with_prefixlen
    except ValueError:
        return None


ipv6_pdf["prefix"] = ipv6_pdf["ipv6"].apply(lambda x: ipv6_to_prefix48(x))
display(ipv6_pdf.head(5))

,ipv6,prefix
0,2620:10a:80ac::45,2620:10a:80ac::/48
1,2001:678:2c:0:194:0:28:7,2001:678:2c::/48
2,2001:678:78:42:ad::53,2001:678:78::/48
3,2001:dce:2000:2::130,2001:dce:2000::/48
4,2001:dce:7000:2::130,2001:dce:7000::/48


In [13]:
responsive_ipv6_pdf = anycast_ipv6_pdf.merge(ipv6_pdf, on="prefix", how="inner")

display(responsive_ipv6_pdf.head(5))
print(len(responsive_ipv6_pdf))

,prefix,AB_ICMPv6,AB_TCPv6,AB_DNSv6,GCD_ICMPv6,GCD_TCPv6,backing_prefix,ASN,locations,ipv6
0,2001:1201:10::/48,1,1,1,3,1,2001:1201:10::/48,27661,"[{'city': 'Vancouver', 'code_country': 'CA', '...",2001:1201:10::1
1,2001:1201::/48,2,2,2,3,2,2001:1201::/48,28498,"[{'city': 'Monterrey', 'code_country': 'MX', '...",2001:1201::1
2,2001:1250:a000::/48,3,2,3,2,1,2001:1250:a000::/44,28511_22894,"[{'city': 'Monterrey', 'code_country': 'MX', '...",2001:1250:a000::1
3,2001:1250:c000::/48,3,2,3,2,1,2001:1250:c000::/44,28511_22894,"[{'city': 'Monterrey', 'code_country': 'MX', '...",2001:1250:c000::1
4,2001:1258::/48,2,2,2,3,2,2001:1258::/32,28499,"[{'city': 'Querétaro', 'code_country': 'MX', '...",2001:1258::1


6017


In [14]:
responsive_ipv6_pdf["ipv6"].to_csv(f"input/responsive_anycast_ipv6_addresses_{ts_v6.year}{ts_v6.month:02d}{ts_v6.day:02d}.csv", index=False, header=False)

# Process output

## Process ZMap output

In [ ]:
def process_zmap_anycast_results(dataset):
    # zmap v4 files are like: zmap_465_20251127123220.csv
    zmap_v4_files = glob(f"results/zmap/dataset={dataset}/**/zmap_*.csv", recursive=True)

    zmap_v4_pdf = pd.DataFrame(columns=["saddr", "port"])
    for zmap_v4_file in zmap_v4_files:
        # extracting the port from the file name
        port = int(zmap_v4_file.split("_")[-2])
        zmap_port_v4_pdf = pd.read_csv(zmap_v4_file)
        zmap_port_v4_pdf["port"] = port
        zmap_v4_pdf = pd.concat([zmap_v4_pdf, zmap_port_v4_pdf[["saddr", "port"]]], ignore_index=True)

    display(
        zmap_v4_pdf.groupby("port").size().reset_index(name="counts").sort_values(by="counts", ascending=False)
    )

    print("unique reponsive IPv4 addresses:", zmap_v4_pdf["saddr"].nunique())

,port,counts
8,443,2428483
4,80,2376009
3,53,683857
5,110,623389
14,995,619655
13,993,619218
1,22,614430
2,25,610767
19,3306,609978
6,143,609429


unique reponsive IPv4 addresses: 2598785


In [ ]:
# far too many results to download and process here.. going to cluster..
process_zmap_anycast_results("tcp-anycast")

In [ ]:
process_zmap_anycast_results("udp-anycast")

,port,counts
0,443,1053948
1,853,1327


unique reponsive IPv4 addresses: 1053974


## Process nmap output

In [152]:
xml_files = glob("../results/nmap/*.xml")

for xml_file in xml_files:
    print(f"Processing {xml_file}...")

    # parse rtt for each host
    tree = ET.parse(xml_file)
    root = tree.getroot()

    host_rtts = {}
    for host in root.findall("host"):
        addr_el = host.find("address")
        ip = addr_el.get("addr") if addr_el is not None else None

        times = host.find("times")
        if times is not None:
            srtt = int(times.get("srtt")) / 1000   # convert to ms
            host_rtts[ip] = srtt

    report = NmapParser.parse_fromfile(xml_file)
    filename = os.path.splitext(os.path.basename(xml_file))[0]
    with open(f"../results/nmap/{filename}.csv", "wt", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["host", "rtt_ms", "port", "protocol", "service", "os", "banner"])
        for host in report.hosts:
            #print(dir(host))
            for svc in host.services:
                writer.writerow([
                    host.address,
                    host_rtts.get(host.address),
                    svc.port,
                    svc.protocol,
                    svc.service,
                    host.os,
                    svc.banner,
                ])



Processing ../results/nmap/nmap_20251204095016_.xml...
Processing ../results/nmap/nmap_20251202093456_.xml...


In [21]:
# https://secwiki.org/w/FAQ_tcpwrapped

In [ ]:
nmap_pdf = pd.read_csv("../results/nmap/nmap_20251202093456_.csv")
display(nmap_pdf)
print(len(nmap_pdf))

nmap_pdf.groupby([ "service"]).size().reset_index(name="counts").sort_values(by="counts", ascending=False)

print("hosts with service", nmap_pdf["host"].nunique())

,host,rtt_ms,port,protocol,service,os,banner
0,3.33.159.7,1.141,1,tcp,tcpmux,Fingerprints:,NaN
1,3.33.159.7,1.141,3,tcp,compressnet,Fingerprints:,NaN
2,3.33.159.7,1.141,4,tcp,NaN,Fingerprints:,NaN
3,3.33.159.7,1.141,6,tcp,NaN,Fingerprints:,NaN
4,3.33.159.7,1.141,7,tcp,echo,Fingerprints:,NaN
...,...,...,...,...,...,...,...
3986,162.159.160.238,1.287,2087,tcp,eli,Fingerprints:,NaN
3987,162.159.160.238,1.287,2095,tcp,nbx-ser,Fingerprints:,NaN
3988,162.159.160.238,1.287,2096,tcp,nbx-dir,Fingerprints:,NaN
3989,162.159.160.238,1.287,8080,tcp,http-proxy,Fingerprints:,NaN


3991


9

In [49]:
nmap_pdf = pd.read_csv("results/nmap/output.csv")
display(nmap_pdf)
print(len(nmap_pdf))

,host,rtt_ms,port,protocol,service,os,banner
0,1.0.0.0,1.212,80,tcp,http,Fingerprints:,product: Cloudflare http proxy
1,1.0.0.0,1.212,443,tcp,https,Fingerprints:,product: cloudflare
2,1.0.0.0,1.212,8080,tcp,http,Fingerprints:,product: Cloudflare http proxy
3,1.0.0.0,1.212,8443,tcp,https-alt,Fingerprints:,product: cloudflare
4,1.1.1.0,1.250,80,tcp,http,Fingerprints:,product: Cloudflare http proxy
5,1.1.1.0,1.250,443,tcp,https,Fingerprints:,product: cloudflare
6,1.1.1.0,1.250,8080,tcp,http,Fingerprints:,product: Cloudflare http proxy
7,1.1.1.0,1.250,8443,tcp,https-alt,Fingerprints:,product: cloudflare
8,1.1.8.8,207.842,80,tcp,http,Fingerprints:,product: nginx version: 1.20.1
9,1.1.8.8,207.842,443,tcp,http,Fingerprints:,product: OpenResty web app server


24


## Process mtr output

In [34]:
mtr_files = glob("results/mtr/*.csv")
for mtr_file in mtr_files:
    print(f"Processing {mtr_file}...")
    # Add your processing code here
    mtr_pdf = pd.read_csv(mtr_file)
    display(mtr_pdf)

Processing results/mtr/mtr_1.12.15.31.csv...


,hostname,Mtr_Version,Start_Time,Status,Host,Hop,Ip,Asn,Loss%,Snt,,Last,Avg,Best,Wrst,StDev
0,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,1,172.18.0.1 (172.18.0.1),AS???,0.0,1,0,0.33,0.33,0.33,0.33,0.0
1,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,2,169.254.169.254 (169.254.169.254),AS???,0.0,1,0,0.20,0.20,0.20,0.20,0.0
2,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,3,vl199-ds1-j2-51917.ams1.constant.com (45.76.40...,AS20473,0.0,1,0,18.60,18.60,18.60,18.60,0.0
3,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,4,10.72.0.5 (10.72.0.5),AS???,0.0,1,0,1.78,1.78,1.78,1.78,0.0
4,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,5,10.72.4.13 (10.72.4.13),AS???,0.0,1,0,0.95,0.95,0.95,0.95,0.0
5,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,6,5-1-40.ear3.amsterdam1.level3.net (213.19.196....,AS3356,0.0,1,0,1.63,1.63,1.63,1.63,0.0
6,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,7,???,AS???,100.0,1,1,0.00,0.00,0.00,0.00,0.0
7,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,8,62.67.67.70 (62.67.67.70),AS3356,0.0,1,0,8.88,8.88,8.88,8.88,0.0
8,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,9,???,AS???,100.0,1,1,0.00,0.00,0.00,0.00,0.0
9,fae9b9eed85a,MTR.0.96,1764079164,OK,1.12.15.31,10,???,AS???,100.0,1,1,0.00,0.00,0.00,0.00,0.0
